# Ensemble Learning

## Setup stage

Mounting Google Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Cloning git repository to access the codebase.

In [ ]:
!git clone -b feat/data-augm --single-branch https://github.com/deadPixelsGreta/xAI-proj-m-ws2526.git

Copying dataset from the Google Drive to the current local disk of VM.

In [ ]:
import shutil
import os
from tqdm import tqdm

source_path = "/content/drive/MyDrive/ImageNetSubset/"
destination_path = "/content/xAI-proj-m-ws2526/datasets/"

# If the destination directory exists, remove it first
if os.path.exists(destination_path):
    print(f"Removing existing directory: {destination_path}")
    shutil.rmtree(destination_path)

# Custom copy function with tqdm
def copytree_with_tqdm(src, dst):
    # Calculate total number of items (files and directories) to copy for tqdm
    total_items = 0
    for dirpath, dirnames, filenames in os.walk(src):
        total_items += len(dirnames) # for directories
        total_items += len(filenames) # for files

    # Ensure the destination root directory exists
    os.makedirs(dst, exist_ok=True)

    with tqdm(total=total_items, unit="item", desc=f"Copying {os.path.basename(src)}") as pbar:
        for dirpath, dirnames, filenames in os.walk(src):
            # Create subdirectories in destination
            relative_path = os.path.relpath(dirpath, src)
            current_dst_dir = os.path.join(dst, relative_path)

            for dirname in dirnames:
                dest_dir = os.path.join(current_dst_dir, dirname)
                os.makedirs(dest_dir, exist_ok=True)
                pbar.update(1)

            # Copy files
            for filename in filenames:
                src_file = os.path.join(dirpath, filename)
                dst_file = os.path.join(current_dst_dir, filename)
                shutil.copy2(src_file, dst_file)
                pbar.update(1)

# Call the custom copy function
copytree_with_tqdm(source_path, destination_path)

Setup the root directory for the project. IMPORTANT for module imports.

In [4]:
import sys, os
from pathlib import Path

def find_project_root(start: Path) -> Path:
    """Walk upward to find the outermost folder containing common project markers."""
    markers = {".git", "requirements.txt", "setup.py", "pyproject.toml"}
    root = None
    for parent in [start, *start.parents]:
        if any((parent / m).exists() for m in markers):
            root = parent  # keep going to prefer the outermost match
    return root or start

# Dynamically get the name of the cloned repository if it exists
cloned_repo_name = "xAI-proj-m-ws2526"
cloned_repo_path = Path.cwd() / cloned_repo_name

# If the cloned repository exists as a subdirectory, change into it
if cloned_repo_path.is_dir():
    os.chdir(cloned_repo_path)

# Now, find the project root from within the repository (or its parent if already there)
ROOT = find_project_root(Path.cwd()).resolve()

# Ensure we are in the identified project root
os.chdir(ROOT)

if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))
print("cwd:", Path.cwd())
print("root on sys.path:", str(ROOT) in sys.path)

cwd: \\studfs01.servinfra.uni-bamberg.de\ba081274\xAI-project\xAI-proj-m-ws2526
root on sys.path: True


In [5]:
# Install dependencies
!pip install -r experiments/requirements.txt --quiet

In [ ]:
Use this path on Windows: C:\Users\ba081274\Downloads\ImageNetSubset\ImageNetSubset\

In [15]:
!python -m experiments.scripts.train --config experiments/configs/default.yaml --model resnet18 --seed 0

RESNET18 Training on ImageNetSubset

Random seed: 0
Device: CUDA (NVIDIA RTX A4000)

 Dataset Summary:
   Training samples: 13000
   Validation samples: 500
   Classes: ['binder', 'coffee_mug', 'computer_keyboard', 'mouse', 'notebook', 'remote_control', 'soup_bowl', 'teapot', 'toilet_tissue', 'wooden_spoon']
   Number of classes: 10

Loading pretrained resnet18 weights...
resnet18 ready with 10 output classes

 Training Configuration:
   Epochs: 5
   Batch size: 16
   Learning rate: 0.001
   Momentum: 0.9
   Weight decay: 0.0001
   Pretrained: True
   Save directory: experiments/checkpoints
   Wandb logging: False

Starting Training

Epoch 1/5
----------------------------------------
   Batch 50/813 | Loss: 1.9356 | Acc: 37.00%
   Batch 100/813 | Loss: 1.6389 | Acc: 51.31%
   Batch 150/813 | Loss: 1.4905 | Acc: 57.71%
   Batch 200/813 | Loss: 1.3992 | Acc: 61.84%
   Batch 250/813 | Loss: 1.3337 | Acc: 64.97%
   Batch 300/813 | Loss: 1.2898 | Acc: 66.85%
   Batch 350/813 | Loss: 1.2587 

---

## Training Stage

In [ ]:
# Train ResNet-18 (via config)
!python -m experiments.scripts.train --config experiments/configs/default.yaml --model resnet18 --seed 0
!python -m experiments.scripts.train --config experiments/configs/default.yaml --model resnet18 --seed 1
!python -m experiments.scripts.train --config experiments/configs/default.yaml --model resnet18 --seed 2


In [ ]:
# Train ResNet-34 (via config)
!python -m experiments.scripts.train --config experiments/configs/default.yaml --model resnet34 --seed 0
!python -m experiments.scripts.train --config experiments/configs/default.yaml --model resnet34 --seed 1
!python -m experiments.scripts.train --config experiments/configs/default.yaml --model resnet34 --seed 2

In [ ]:
# Train EfficientNet-B0 (via config)
!python -m experiments.scripts.train --config experiments/configs/default.yaml --model efficientnet_b0 --seed 0
!python -m experiments.scripts.train --config experiments/configs/default.yaml --model efficientnet_b0 --seed 1
!python -m experiments.scripts.train --config experiments/configs/default.yaml --model efficientnet_b0 --seed 2


---

## Validation Stage

In [ ]:
# Run ensemble evaluation on a dataset
!python -m experiments.scripts.inference --evaluate --data-dir datasets --wandb

In [ ]:
# Upload a test image or use sample
!python -m experiments.scripts.inference --image datasets/test_image.jpg --show-individual

---

## Script for saving best models in Google Drive

In [ ]:
# Define source and destination paths
# We check experiments/checkpoints (relative to project root) first
source_dir = Path("experiments/checkpoints")

# Check for fallback paths if the default doesn't exist (e.g., if strictly using /chechpoints)
if not source_dir.exists():
    if Path("/chechpoints").exists(): # Handling the specific path mentioned
        source_dir = Path("/chechpoints")
    elif Path("/checkpoints").exists(): # Handling potential typo correction
        source_dir = Path("/checkpoints")

# Destination folder on the mounted drive
# You can change "saved_checkpoints" to your preferred folder name
dest_dir = Path("/content/drive/MyDrive/saved_checkpoints")

print(f"Source Directory: {source_dir}")
print(f"Destination Directory: {dest_dir}")

# Create destination directory if it doesn't exist
os.makedirs(dest_dir, exist_ok=True)

# Copy .pth files
if source_dir.exists():
    pth_files = list(source_dir.glob("*.pth"))
    
    if not pth_files:
        print("No .pth files found in source directory.")
    else:
        print(f"Found {len(pth_files)} .pth files to copy.")
        
        for file_path in pth_files:
            try:
                shutil.copy2(file_path, dest_dir / file_path.name)
                print(f"Successfully copied: {file_path.name}")
            except Exception as e:
                print(f"Error copying {file_path.name}: {e}")
else:
    print(f"Source directory {source_dir} not found. Please check the path.")